# Trabalho Grau B - Reconhecimento de imagem e transfer learning

## Integrantes

- Arthur Schallenberger
- Giovani de Souza
- Leonardo Fronza
- Renan Milech Pereira

---

**Docente:** Prof. Gabriel de Oliveira Ramos

## 2.1 Descrição do Problema e Dataset

### 2.1.1 Descrição do Problema

O problema abordado neste trabalho é a **classificação multiclasse de imagens de bolas esportivas**. Dado um conjunto de imagens coloridas, o objetivo é treinar um modelo capaz de identificar corretamente a qual modalidade esportiva pertence a bola presente na imagem, dentre 15 categorias distintas.

Trata-se de um problema de **visão computacional supervisionada**, em que cada imagem possui um único rótulo associado (classe da bola). O desafio central está na variação visual entre as classes — algumas bolas possuem formas, texturas e padrões muito distintos (como a bola de futebol americano e a de tênis de mesa), enquanto outras apresentam características visuais próximas (como bola de cricket, hockey e tênis), exigindo que o modelo aprenda representações discriminativas e generalizáveis.

A escolha desse domínio é motivada pela disponibilidade de dados rotulados e pela aplicabilidade prática em sistemas de análise de transmissões esportivas, arbitragem automatizada e catalogação de conteúdo multimídia.

---

### 2.1.2 Dataset

O dataset utilizado é o **Sports Ball Image Recognition**, disponível publicamente na plataforma [Kaggle](https://www.kaggle.com/). Ele é composto por imagens JPEG de bolas de 15 modalidades esportivas diferentes, já organizadas em subpastas por classe e divididas entre conjuntos de treino e teste.

#### Estrutura de diretórios

```
archive/
├── train/
│   ├── american_football/
│   ├── baseball/
│   ├── ...
│   └── volleyball/
└── test/
    ├── american_football/
    ├── baseball/
    ├── ...
    └── volleyball/
```

#### Classes e distribuição de imagens

O dataset contém **15 classes**, com a seguinte distribuição por split:

| Classe               | Treino | Teste | Total |
|----------------------|--------|-------|-------|
| american_football    | 384    | 96    | 480   |
| baseball             | 400    | 100   | 500   |
| basketball           | 340    | 86    | 426   |
| billiard_ball        | 646    | 162   | 808   |
| bowling_ball         | 440    | 111   | 551   |
| cricket_ball         | 581    | 146   | 727   |
| football             | 604    | 151   | 755   |
| golf_ball            | 549    | 138   | 687   |
| hockey_ball          | 530    | 133   | 663   |
| hockey_puck          | 390    | 98    | 488   |
| rugby_ball           | 493    | 124   | 617   |
| shuttlecock          | 429    | 108   | 537   |
| table_tennis_ball    | 620    | 156   | 776   |
| tennis_ball          | 490    | 123   | 613   |
| volleyball           | 432    | 109   | 541   |
| **Total**            | **7.328** | **1.841** | **9.169** |

A divisão treino/teste segue uma proporção aproximada de **80%/20%**, padrão comum em benchmarks de visão computacional.

#### Características das imagens

- **Formato:** JPEG (`.jpg`)
- **Resolução:** variada — as imagens originais possuem dimensões heterogêneas (desde 225×225 até resoluções maiores como 1920×1080 e 2048×1152)
- **Canais:** RGB (3 canais de cor)
- **Pré-processamento necessário:** todas as imagens serão redimensionadas para **224×224 pixels** antes de serem alimentadas nos modelos, por ser o formato esperado pelas arquiteturas EfficientNetB0 e MobileNetV2, e também utilizado na CNN própria para padronização

#### Balanceamento das classes

O dataset apresenta um **leve desbalanceamento** entre as classes. A classe com mais amostras (*billiard_ball*) possui 808 imagens, enquanto a menor (*basketball*) possui 426 — uma razão de aproximadamente 1,9×. Esse nível de desbalanceamento é considerado moderado e não exige técnicas agressivas de reamostragem, mas será monitorado durante o treinamento por meio de métricas por classe (precisão, revocação e F1-score).


## 2.2 Análise das Classes e Dados

O conjunto de dados utilizado corresponde ao repositório 'sports-balls-multiclass-image-classification' (Kaggle), organizado em subdiretórios por classe e contendo amostras para treino e teste em `DATASET/archive`.

Visão geral:
- Número de classes: 15.
- Total de imagens (conjunto de treino): 7.328.
- Distribuição por classe: variação moderada entre aproximadamente 340 e 646 imagens por classe, indicando um desbalanceamento leve.
- Resolução média das imagens: aproximadamente 606×509 pixels.
- Integridade: não foram detectadas imagens corrompidas no conjunto de treino analisado.

Organização dos dados:
Os arquivos encontram-se estruturados em subpastas por rótulo (`DATASET/archive/train/<classe>`), facilitando a associação direta imagem→rótulo e a replicabilidade dos experimentos.

Observações qualitativas:
Há variação significativa nas condições de aquisição (fundos, iluminação e escalas), bem como diferenças no tamanho relativo do objeto em relação ao quadro. Algumas classes apresentam similaridades visuais que podem dificultar a discriminação, por exemplo entre `football` e `rugby_ball`, ou entre `hockey_ball` e `hockey_puck`. Essas características implicam maior exigência na capacidade do modelo de extrair representações discriminativas.

Considerações metodológicas:
A heterogeneidade das imagens e as semelhanças inter-classes sugerem a adoção de estratégias de modelagem que privilegiem a generalização e a robustez a variações geométricas e fotométricas. A avaliação deve contemplar medidas por classe (precisão, revocação e F1-score) e análise da matriz de confusão, de modo a evidenciar comportamentos assimétricos entre classes.

Implicações para seleção de modelos e avaliação:
Modelos pré-treinados (por exemplo, arquiteturas amplamente utilizadas em visão computacional) são adequados para exploração inicial do problema, dado o volume de dados e a diversidade visual. Também é pertinente considerar abordagens de fine-tuning e estratégias de regularização que mitiguem overfitting em classes com menor número de amostras. Caso se observe queda de desempenho concentrada em algumas classes, técnicas de ponderação e análise dirigida de falsos positivos devem ser empregadas para diagnosticar as causas subjacentes.

Resumo conclusivo:
O conjunto de dados apresenta qualidade e diversidade suficientes para a investigação de modelos de classificação multiclasses. A análise qualitativa realizada aponta para desafios esperados (variação de aquisição e classes visualmente próximas) que devem ser explicitados no relatório final e considerados na definição do protocolo experimental.


## 2.3 Pré-processamento

O pré-processamento padroniza as imagens antes do treinamento e aplica augmentação para melhorar a generalização dos modelos. As etapas adotadas são descritas a seguir.

### 2.3.1 Redimensionamento

Todas as imagens são redimensionadas para **224 × 224 pixels**, resolução exigida pelas arquiteturas EfficientNetB0 e MobileNetV2 (entrada padrão treinada com ImageNet) e adotada também na CNN própria para uniformizar os experimentos. O redimensionamento é realizado durante o carregamento via `ImageDataGenerator`, sem modificar os arquivos originais.

### 2.3.2 Normalização

A normalização é aplicada de forma específica para cada modelo, pois cada arquitetura foi treinada com uma escala de pixels diferente:

| Modelo         | Função de normalização                             | Intervalo de saída |
|----------------|----------------------------------------------------|--------------------|
| CNN própria    | `rescale = 1 / 255`                                | [0, 1]             |
| EfficientNetB0 | `efficientnet.preprocess_input`                    | Centrado em 0 (ImageNet) |
| MobileNetV2    | `mobilenet_v2.preprocess_input`                    | [−1, 1]            |

O uso das funções `preprocess_input` específicas de cada rede garante compatibilidade com os pesos pré-treinados no ImageNet, evitando degradação de desempenho na fase de *transfer learning*.

### 2.3.3 Divisão Treino / Validação / Teste

O dataset já fornece uma separação explícita entre `train/` e `test/`. Para o conjunto de validação, são reservados **10 %** das amostras de treino via `validation_split`, com semente fixa (`SEED = 42`) para reprodutibilidade:

| Conjunto  | Origem                  | Imagens (aprox.) |
|-----------|-------------------------|-----------------|
| Treino    | `archive/train/` (90 %) | ≈ 6.595          |
| Validação | `archive/train/` (10 %) | ≈ 733            |
| Teste     | `archive/test/`         | 1.841            |

### 2.3.4 Augmentação de Dados

As transformações abaixo são aplicadas **apenas nas imagens de treino**. Os conjuntos de validação e teste recebem somente a normalização, para que a avaliação seja feita com amostras o mais próximas possível das condições reais.

| Transformação       | Parâmetro          | Justificativa                                        |
|---------------------|--------------------|------------------------------------------------------|
| Rotação             | ±15°               | Bolas podem aparecer em qualquer orientação           |
| Deslocamento H / V  | ±10 %              | Variações de enquadramento na captura                |
| Flip horizontal     | Ativado            | Simetria presente na maioria das bolas               |
| Zoom                | ±10 %              | Variação de escala do objeto no quadro               |
| Brilho              | [0,8 – 1,2]        | Condições variadas de iluminação                     |

Essas augmentações introduzem diversidade artificial sem distorcer as características discriminativas das bolas (forma, textura e padrão de cor), contribuindo para reduzir o *overfitting*, especialmente nas classes com menor número de amostras.


## 2.4 Arquitetura das Redes Neurais

Para o trabalho, foram consideradas duas abordagens complementares: uma CNN construída do zero, usada como linha de base, e redes pré-treinadas com *transfer learning*, especialmente EfficientNetB0 e MobileNetV2 (que são redes leves e eficientes e que uma delas será escolhida para o transfer learning).

### 2.4.1 CNN própria

A CNN própria foi pensada para ser simples, estável e fácil de interpretar. A arquitetura proposta segue a lógica de extração progressiva de características:

- **Camada de entrada:** imagens redimensionadas para um formato fixo, como 224 x 224 x 3.
- **Blocos convolucionais:** 3 blocos com convoluções 2D, ativação ReLU e *padding* igual.
- **Pooling:** *MaxPooling2D* após cada bloco convolucional para reduzir dimensionalidade e manter as informações mais relevantes.
- **Regularização:** *Dropout* entre os blocos e antes da saída para reduzir *overfitting*.
- **Classificação final:** camadas densas com *softmax* na saída, uma neurônio por classe.

Uma configuração coerente para essa CNN é:

- Bloco 1: 32 filtros, convolução 3 x 3, ReLU, MaxPooling
- Bloco 2: 64 filtros, convolução 3 x 3, ReLU, MaxPooling
- Bloco 3: 128 filtros, convolução 3 x 3, ReLU, MaxPooling
- *Flatten* ou *GlobalAveragePooling2D*
- *Dense* final com *softmax*

Essa estrutura é suficiente para capturar padrões visuais básicos e serve como referência para comparar com as redes pré-treinadas.

### 2.4.2 Transfer learning

A rede de *transfer learning* escolhida para o experimento principal foi a **EfficientNetB0**, por apresentar bom equilíbrio entre desempenho e custo computacional. A **MobileNetV2** foi mantida como comparação leve, útil quando a prioridade é reduzir parâmetros e acelerar inferência.

A estratégia adotada para a EfficientNetB0 (experimento principal):

1. Carregar os pesos pré-treinados no ImageNet.
2. Congelar a base convolucional nas primeiras etapas.
3. Adicionar uma cabeça de classificação específica para as classes do conjunto de dados.
4. Se necessário, liberar parte das últimas camadas para *fine-tuning*.

### 2.4.3 Justificativa das escolhas

As escolhas arquiteturais foram feitas considerando o tamanho do conjunto de dados e o objetivo de classificação de imagens esportivas:

- A **CNN própria** funciona como baseline e permite avaliar o quanto o problema pode ser resolvido sem conhecimento prévio transferido.
- **EfficientNetB0** tende a oferecer melhor relação entre profundidade, eficiência e generalização, sendo uma boa candidata para maior acurácia.
- **MobileNetV2** é uma alternativa mais leve, com menor custo de processamento, útil para comparação e para cenários com limitação de recursos.
- O uso de **Dropout** e de *MaxPooling* ajuda a controlar o sobreajuste e a reduzir o tamanho das representações intermediárias.
- A ativação **ReLU** é adequada por ser simples, eficiente e amplamente usada em CNNs modernas.

### 2.4.4 Comparação das arquiteturas

| Arquitetura | Número de camadas | Convoluções | Pooling | Dropout | Função de ativação | Vantagem principal |
| --- | --- | --- | --- | --- | --- | --- |
| CNN própria | 3 blocos convolucionais + classificadores | 3 x 3 | MaxPooling2D | Sim | ReLU / Softmax | Baseline simples e interpretável |
| EfficientNetB0 | Backbone pré-treinado + cabeça densa | Convoluções otimizadas pela família EfficientNet | GlobalAveragePooling2D ou pooling implícito | Sim | Swish/ReLU + Softmax | Melhor equilíbrio entre desempenho e custo |
| MobileNetV2 | Backbone pré-treinado + cabeça densa | Convoluções separáveis | GlobalAveragePooling2D | Sim | ReLU6 + Softmax | Menor custo computacional |

### 2.4.5 Diagrama simplificado

```mermaid
flowchart LR
    A[Imagem de entrada\n224 x 224 x 3] --> B{Estratégia}
    B --> C[CNN própria\nConv 3x3 + ReLU\nMaxPooling + Dropout]
    B --> D[EfficientNetB0\nbase congelada + cabeça densa]
    B --> E[MobileNetV2\nbase congelada + cabeça leve]
    C --> F[Softmax\nclassificação]
    D --> F
    E --> F
```

Em resumo, a CNN própria fornece a linha de base do estudo, enquanto EfficientNetB0 e MobileNetV2 representam as melhores alternativas de *transfer learning* para comparar desempenho, robustez e custo de execução.